In [12]:
%load_ext autoreload
%autoreload 2

import numpy as np
import time
import pickle
import os
import subprocess
import matplotlib.pyplot as plt
from scipy.stats import norm

from e_1_run_cvae import compare_prices
from e_2_CVAE import *


# global var
S0 = 1.0
K = 1.0
r = 0.03
sigma = np.sqrt(0.05)
T = 1.5
BS_eta = (S0, K, r, sigma, T)
# S0, K, r, kappa, theta, xi, rho, Y0, T = Hes_eta
Hes_eta = (S0, K, r, 2, 0.05, 0.5, -0.7, 0.05, T)

B = 0.8 # down-and-out must B < S0 and B < K
opt_type = 'call' # call or put
barr_type = 'van' # van or barr
model_type = 'bs' # hes or bs

if not((B < S0) & (B < K)):
    raise ValueError("down-and-out : B should be smaller than S0 and K")

if not(opt_type == 'call' or  opt_type == 'put'):
    raise ValueError("option_type must be 'call' or 'put'")

if not(barr_type == 'van' or  barr_type == 'barr'):
    raise ValueError("barr_type must be 'van' or 'barr'")

if not(model_type == 'hes' or  model_type == 'bs'):
    raise ValueError("model_type must be 'hes' or 'bs'")

# cvae training settings
if model_type == 'hes':
    if barr_type == 'barr':
        if opt_type == 'call':
            bench_price = 0.115733
        else: # put
            bench_price = 0.005170
    else: # van
        if opt_type == 'call':
            bench_price = 0.124491
        else: # put
            bench_price = 0.080488

elif model_type == 'bs':
    if barr_type == 'barr':
        if opt_type == 'call':
            bench_price = 0.123493
        else: # put
            bench_price = 0.009535
    else: # van
        if opt_type == 'call':
            bench_price = 0.129944
        else: # put
            bench_price = 0.085942

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [30]:
import os, subprocess, sys

env = os.environ.copy()
env["CUDA_VISIBLE_DEVICES"] = "0,1,2,3,4,5,6,7"

code = """
import torch, os
print("CUDA_VISIBLE_DEVICES =", os.environ.get("CUDA_VISIBLE_DEVICES"))
print("device_count =", torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    print(i, torch.cuda.get_device_name(i))
"""

subprocess.run([sys.executable, "-c", code], env=env, check=True)

CUDA_VISIBLE_DEVICES = 0,1,2,3,4,5,6,7
device_count = 7
0 NVIDIA GeForce GTX 1080 Ti
1 NVIDIA GeForce GTX 1080 Ti
2 NVIDIA GeForce GTX 1080 Ti
3 NVIDIA GeForce GTX 1080 Ti
4 NVIDIA GeForce GTX 1080 Ti
5 NVIDIA GeForce GTX 1080 Ti
6 NVIDIA GeForce GTX 1080 Ti


/home/ajoufe/anaconda3/lib/python3.9/site-packages/torch/cuda/__init__.py:497: UserWarning: Can't initialize NVML
  warnings.warn("Can't initialize NVML")


CompletedProcess(args=['/home/ajoufe/anaconda3/bin/python', '-c', '\nimport torch, os\nprint("CUDA_VISIBLE_DEVICES =", os.environ.get("CUDA_VISIBLE_DEVICES"))\nprint("device_count =", torch.cuda.device_count())\nfor i in range(torch.cuda.device_count()):\n    print(i, torch.cuda.get_device_name(i))\n'], returncode=0)

In [31]:
dim_z       = 8 # 12
hidden_dims = [128, 128, 64] # [128, 128, 64], [256, 256, 128], [512, 512, 256]
batch_size  = 4096 
n_epochs    = 5 # loss 수렴할 때까지 
lr          = 1e-3 # 3e-4, 5e-4
beta        = 1.0

n_samples = 10000 # n_samples= 1k, 10k, 100k
if n_samples % 2 != 0:
    raise ValueError("n_samples should be an even number for antithetic sampling")

if model_type == 'hes':
    test_etas = [0.03, 2.0,  0.05, 0.5, -0.7, 0.05, 1.5]
    eta_keys  = ['r', 'lambda', 'v_bar', 'xi', 'rho', 'Y0', 'T']
else: # model_type = 'bs'
    test_etas = [r, sigma, T]
    eta_keys  = ['r', 'sigma', 'T']

gpu_ids = "0,1,2,3,4,5,6" # 7 GPU problem
nproc = len(gpu_ids.split(","))

resume_path = None
save_path = f"cvae_{model_type}_{barr_type}_{dim_z}_{hidden_dims[0]}_{batch_size}_ddp.pt"
hidden_dims_arg = ",".join(map(str, hidden_dims))

In [ ]:
env = os.environ.copy()
env["CUDA_VISIBLE_DEVICES"] = gpu_ids
env["NCCL_DEBUG"] = "INFO"
env["TORCH_DISTRIBUTED_DEBUG"] = "DETAIL"
env["NCCL_ASYNC_ERROR_HANDLING"] = "1"
env["NCCL_P2P_DISABLE"] = "1"
env["NCCL_IB_DISABLE"] = "1"

cmd = [
    "torchrun",
    f"--nproc_per_node={nproc}",
    "f_run_cvae_ddp.py",
    "--model-type", model_type,
    "--barr-type", barr_type,
    "--dim-z", str(dim_z),
    "--hidden-dims", hidden_dims_arg,
    "--batch-size", str(batch_size),
    "--epochs", str(n_epochs),
    "--lr", str(lr),
    "--beta", str(beta),
    "--save-path", str(save_path),
    "--num-workers", "2",
]

if resume_path is not None:
    cmd += ["--resume-path", str(resume_path)]

subprocess.run(cmd, env=env, check=True)

*****************************************
Setting OMP_NUM_THREADS environment variable for each process to be 1 in default, to avoid your system being overloaded, please further tune the variable for optimal performance in your application as needed. 
*****************************************
/home/ajoufe/anaconda3/lib/python3.9/site-packages/torch/cuda/__init__.py:497: UserWarning: Can't initialize NVML
  warnings.warn("Can't initialize NVML")
/home/ajoufe/anaconda3/lib/python3.9/site-packages/torch/cuda/__init__.py:497: UserWarning: Can't initialize NVML
  warnings.warn("Can't initialize NVML")
/home/ajoufe/anaconda3/lib/python3.9/site-packages/torch/cuda/__init__.py:497: UserWarning: Can't initialize NVML
  warnings.warn("Can't initialize NVML")
/home/ajoufe/anaconda3/lib/python3.9/site-packages/torch/cuda/__init__.py:497: UserWarning: Can't initialize NVML
  warnings.warn("Can't initialize NVML")
/home/ajoufe/anaconda3/lib/python3.9/site-packages/torch/cuda/__init__.py:497: UserWar

eta min/max 계산 중
eta min/max 계산 중
eta min/max 계산 중
eta min/max 계산 중
eta min/max 계산 중
eta min/max 계산 중
eta min/max 계산 중
계산 완료

계산 완료

계산 완료

계산 완료

계산 완료

계산 완료

계산 완료

ajoufe1:33762:33762 [0] NCCL INFO Bootstrap : Using enp13s0f0:192.168.0.15<0>
ajoufe1:33762:33762 [0] NCCL INFO NET/Plugin : No plugin found (libnccl-net.so), using internal implementation
ajoufe1:33762:33762 [0] NCCL INFO cudaDriverVersion 12050
NCCL version 2.14.3+cuda11.7
ajoufe1:33765:33765 [3] NCCL INFO cudaDriverVersion 12050
ajoufe1:33766:33766 [4] NCCL INFO cudaDriverVersion 12050
ajoufe1:33764:33764 [2] NCCL INFO cudaDriverVersion 12050
ajoufe1:33768:33768 [6] NCCL INFO cudaDriverVersion 12050
ajoufe1:33767:33767 [5] NCCL INFO cudaDriverVersion 12050
ajoufe1:33763:33763 [1] NCCL INFO cudaDriverVersion 12050
ajoufe1:33762:33842 [0] NCCL INFO NCCL_IB_DISABLE set by environment to 1.
ajoufe1:33762:33842 [0] NCCL INFO NET/Socket : Using [0]enp13s0f0:192.168.0.15<0>
ajoufe1:33762:33842 [0] NCCL INFO Using network Soc

In [ ]:
# compare_price
ckpt = torch.load(save_path, map_location="cuda", weights_only=False)

cvae = CVAE(
    dim_x=ckpt["dim_x"],
    dim_eta=ckpt["dim_eta"],
    dim_z=ckpt["dim_z"],
    hidden_dims=ckpt["hidden_dims"],
).to("cuda")

cvae.load_state_dict(ckpt["model_state"])
cvae.eval()

loss_history = ckpt["loss_history"]
eta_min = ckpt["eta_min"]
eta_max = ckpt["eta_max"]

In [ ]:
if barr_type == 'barr':
    compare_prices(cvae, B, K, eta_min, eta_max, test_etas, eta_keys, bench_price, 
                opt_type, barr_type, n_samples) 
elif barr_type == 'van':
    compare_prices(cvae, None, K, eta_min, eta_max, test_etas, eta_keys, bench_price, 
                opt_type, barr_type, n_samples)